In [9]:
import pandas as pd
import numpy as np
import sklearn
#pd.set_option('display.max_colwidth', -1)
import re


In [10]:
labeled_path = "../Abraham/Reports_IBD_Cohort_080320.csv"
labeled_data = pd.read_csv(labeled_path, encoding = "ISO-8859-1",header=0)

In [11]:
initial_copy = labeled_data.copy()
initial_copy


,mrn,service_date,result
0,12181,2013-08-18 00:00:00.000,IMPRESSION: 1. Dilated proximal and decompress...
1,18163,2013-08-20 00:00:00.000,1. Multiple circumscribed rounded low density...
2,68012,2000-11-13 00:00:00.000,Visit # 4055251 XRAY DATE: 11-13...
3,68012,2007-12-21 00:00:00.000,ABDOMINAL PAIN CT OF THE ABDOMEN AND PELVIS: 1...
4,68144,2018-04-04 00:00:00.000,1. Stool-filled colon. Marked rectal distenti...
...,...,...,...
13509,78119306,2020-06-05 00:00:00.000,1. Status post partial colectomy with left low...
13510,78167572,2020-07-28 00:00:00.000,1. Fecalization of intraluminal contents throu...
13511,78174358,2020-07-24 00:00:00.000,CT ENTEROGRAPHY 7/24/2020 9:55 AM CLINICAL HI...
13512,78473037,2020-06-28 00:00:00.000,1. . Appendix dilation up to 10 mm with mucosa...


In [12]:
def process_note(note):
    note = re.sub(r'.*REPORT', '', note) # Remove beginning 
    note = re.sub(r'\d./\d./\d*', '', note) # Remove dates
    note = re.sub(r'\d:\d+(.AM|.PM)', '', note) # Remove times
    note = re.sub(r'\d{6,}', '', note) # Remove ID numbers
    #note = re.sub(r'\d', '', note) # Remove numbers in general
    #note = re.sub(r"(?=_____________________________).*", "", note)
    #note = re.sub(r"Ê", "", note)
    return note

In [13]:
#create separate dataframe with processed notes
initial_copy['result'] = initial_copy['result'].apply(process_note)
initial_copy.to_csv("Processed.csv", index=False)

In [14]:
copy2 = initial_copy.copy()
copy3 = initial_copy.copy()

In [15]:
#create indicator variable in separate df for "clinical history/indications"
def indic_ch(note):
    note["'clinical history' presence"] = note['result'].str.contains('CLINICAL HISTORY:', regex=True)
    note["'clinical data' presence"] = note['result'].str.contains('CLINICAL DATA:', regex=True)
    note['Clinical History Indicators'] = note['result'].str.extract(r'CLINICAL HISTORY:(.*?)COMPARISON|TECHNIQUE|ADDITIONAL HISTORY|FINDING')
    note['Clinical Data Indicators'] = note['result'].str.extract(r'CLINICAL DATA:(.*?)COMPARISON|TECHNIQUE|ADDITIONAL HISTORY|FINDING')
    note = note.fillna('')
    note['Clinical Indicators (Combined)'] = note['Clinical History Indicators'].str.cat(note['Clinical Data Indicators'],sep=" ")
    #note['indications'] = note['Clinical History Indicators'] + note['Clinical Data Indicators']
    note["'technique' presence"] = note['result'].str.contains('TECHNIQUE:', regex=True)
    note['technique indicators'] = note['result'].str.extract(r'TECHNIQUE:(.*?):')
    note = note.fillna('')
    note["'oral contrast' presence"] = note['result'].str.contains('oral [a-z\s]*?contrast', regex=True)
    note["'rectal contrast' presence"] = note['result'].str.contains('rectal [a-z\s]*?contrast', regex=True)
    note["'MRI' presence"] = note['result'].str.contains('MRI', regex=True)
    note["'enterography' presence"] = note['result'].str.contains('enterography', regex=True)
    note["'enterography' header presence"] = note['result'].str.contains('ENTEROGRAPHY', regex=True)
    note["'MR enterography' header presence"] = note['result'].str.contains('MR ENTEROGRAPHY', regex=True)
    note['CT'] = 1 - (note["'MRI' presence"])
    note['CT'] = note['CT'].astype(bool)
    
    return note
with_indic = indic_ch(copy3)
with_indic.to_csv("notes_indicators.csv")

In [16]:
def indic_technique(note):
    note["'IV Contrast' presence"] = note['technique indicators'].str.contains('[Ii][Vv] contrast|[Ii]ntravenous [Cc]ontrast|[Oo]mnipaque|[Ee]nterography', regex=True)
    for index, row in note.iterrows():
#         if row["'IV Contrast' presence"].isnull():
#             note = note.drop(index)
        if row["'IV Contrast' presence"] != 1:
            #doesn't actually drop
            note = note.drop(index)
    #return df
    return note
with_IV = indic_technique(with_indic)
IV_indic = with_IV[['mrn', 'service_date', 'result', 'Clinical Indicators (Combined)', 'technique indicators', "'IV Contrast' presence", "'oral contrast' presence", "'rectal contrast' presence", "'MRI' presence", 'CT', "'enterography' presence", "'enterography' header presence", "'MR enterography' header presence"]]
#IV_indic = IV_indic["'IV Contrast' presence"].dropna().astype('int32')
#IV_indic["'IV Contrast' presence" == True]
IV_indic
IV_indic.to_csv("IV_indic.csv")

In [17]:
IV_indic["'ulcerative' presence"] = [bool(re.search(r"[Uu]lcerative", x)) for x in IV_indic["Clinical Indicators (Combined)"]]
IV_indic["'colitis' presence"] = [bool(re.search(r"[Cc]olitis", x)) for x in IV_indic["Clinical Indicators (Combined)"]]
IV_indic["'proctitis' presence"] = [bool(re.search(r"[Pp]roctitis", x)) for x in IV_indic["Clinical Indicators (Combined)"]]
IV_indic["'\sUC\s' presence"] = [bool(re.search(r"UC\s", x)) for x in IV_indic["Clinical Indicators (Combined)"]]
IV_indic["'Crohn' presence"] = [bool(re.search(r"[Cc]rohn", x)) for x in IV_indic["Clinical Indicators (Combined)"]]
IV_indic["'Chron' presence"] = [bool(re.search(r"[Cc]hron", x)) for x in IV_indic["Clinical Indicators (Combined)"]]
IV_indic["'rectosigmoiditis' presence"] = [bool(re.search(r"rectosigmoiditis", x)) for x in IV_indic["Clinical Indicators (Combined)"]]
IV_indic["'proc-omy' presence"] = [bool(re.search(r"proc[a-z]+?omy", x)) for x in IV_indic["Clinical Indicators (Combined)"]]
IV_indic["'ileo-omy' presence"] = [bool(re.search(r"ileo[a-z]+?omy", x)) for x in IV_indic["Clinical Indicators (Combined)"]]
IV_indic["'without' presence (in Technique)"] = [bool(re.search(r"without", x)) for x in IV_indic["technique indicators"]]
IV_indic["'no' presence (in Technique)"] = [bool(re.search(r"\sno\s", x)) for x in IV_indic["technique indicators"]]
IV_indic["'without' before 'IV contrast'-like terms (in Technique)"] = [bool(re.search(r"(\bwithout\b)(?:[^.]|\.\s)*(\b[Ii][Vv] contrast|[Ii]ntravenous [Cc]ontrast|[Oo]mnipaque|[Ee]nterography\b)", x)) for x in IV_indic["technique indicators"]]
IV_indic.to_csv("IV_indic.csv")



/opt/ars-virtenv/lib/python3.6/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/opt/ars-virtenv/lib/python3.6/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/opt/ars-virtenv/lib/python3.6/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

In [18]:
post_2012 = IV_indic.copy()
#post_2012['service_date'].astype(str)
pd.to_datetime(post_2012['service_date'])
post_2012 = post_2012[post_2012['service_date'] >= '2012-01-01']
post_2012.to_csv('post_2012.csv')

In [19]:
# def inferred_pos_indic(note):
#     df = note[["mrn", "service_date", "Clinical Indicators (Combined)"]].copy()
#     notecopy = note[["mrn", "service_date", "Clinical Indicators (Combined)", "'ulcerative' presence",
#                    "'colitis' presence", "'proctitis' presence", "'\sUC\s' presence", "'Crohn' presence", 
#                     "'Chron' presence", "'rectosigmoiditis' presence"]]
#     col_list = list(notecopy)
#     col_list.remove('mrn')
#     col_list.remove('Clinical Indicators (Combined)')
#     col_list.remove('service_date')
#     col_list.remove("'Crohn' presence")
#     col_list.remove("'Chron' presence")
#     not_list = list(note[["'Crohn' presence", "'Chron' presence"]])
#     notecopy["Prediction"] =  notecopy[col_list].sum(axis=1)
#     notecopy["Chron or Crohn"] =  notecopy[not_list].sum(axis=1)
#     #predictions = []
#     #for x in notecopy["Prediction"]:
#         #if x > 0:
#          #   predictions.append("Active")
#         #if x == 0:
#          #   predictions.append("Inactive")
#     #df["predictions"] = ["Active" if x > 0 and y == 0 else "Inactive" for x in notecopy["Prediction"] for y in notecopy["Chron or Crohn"]]
#     df['predictions'] = np.where(notecopy[col_list].sum(axis=1) > 0, 'Positive', 'Negative')
#     return df
# b = inferred_pos_indic(pos_df_indic)
# b.to_csv("inferred_pos_indic.csv")

In [20]:
# def inferred_neg_indic(note):
#     df = note[["mrn", "service_date", "Clinical Indicators (Combined)"]].copy()
#     notecopy = note[["mrn", "service_date", "Clinical Indicators (Combined)", "'ulcerative' presence",
#                    "'colitis' presence", "'proctitis' presence", "'\sUC\s' presence", "'Crohn' presence", 
#                     "'Chron' presence", "'rectosigmoiditis' presence", "'proc-omy' presence","'ileo-omy' presence"]]
#     col_list = list(notecopy)
#     col_list.remove('mrn')
#     col_list.remove('Clinical Indicators (Combined)')
#     col_list.remove('service_date')
#     col_list.remove("'Crohn' presence")
#     col_list.remove("'Chron' presence")
#     not_list = list(note[["'Crohn' presence", "'Chron' presence", "'proc-omy' presence","'ileo-omy' presence"]])
#     notecopy["Prediction"] =  notecopy[col_list].sum(axis=1)
#     notecopy["Chron or Crohn"] =  notecopy[not_list].sum(axis=1)
#     #predictions = []
#     #for x in notecopy["Prediction"]:
#         #if x > 0:
#          #   predictions.append("Active")
#         #if x == 0:
#          #   predictions.append("Inactive")
#     #df["predictions"] = ["Inactive" if y > 0 else "Active" for y in notecopy["Prediction"] for y = notecopy["Chron or Crohn"]]
#     df['predictions'] = np.where(notecopy[not_list].sum(axis=1) > 0, 'Negative', 'Positive')
#     return df
# c = inferred_neg_indic(neg_df_indic)
# c
# #c.to_csv("inferred_neg_indic.csv")

In [21]:
post_2012_copy_pred = post_2012.copy()
def pred_indic(df):
    neg = df["'Crohn' presence"] + df["'Chron' presence"] + df["'proc-omy' presence"] + df["'ileo-omy' presence"] > 0
    pos = df["'ulcerative' presence"] + df["'colitis' presence"] + df["'proctitis' presence"] + df["'\sUC\s' presence"] + df["'rectosigmoiditis' presence"] > 0
    conditions = [
    (neg),
    (pos)
    ]

    # create a list of the values we want to assign for each condition
    values = ['Negative', 'Positive']

    # create a new column and use np.select to assign values to it using our lists as arguments
    df['prediction'] = np.select(conditions, values)
    return df
post_2012_copy_pred = pred_indic(post_2012_copy_pred)
post_2012_copy_pred.to_csv('post_2012_copy_pred.csv')

/opt/ars-virtenv/lib/python3.6/site-packages/pandas/core/computation/expressions.py:194: UserWarning: evaluating in Python space because the '+' operator is not supported by numexpr for the bool dtype, use '|' instead
  op=op_str, alt_op=unsupported[op_str]


In [22]:
post_2012_copy_pred[post_2012_copy_pred['prediction'] == 'Positive']

,mrn,service_date,result,Clinical Indicators (Combined),technique indicators,'IV Contrast' presence,'oral contrast' presence,'rectal contrast' presence,'MRI' presence,CT,...,'\sUC\s' presence,'Crohn' presence,'Chron' presence,'rectosigmoiditis' presence,'proc-omy' presence,'ileo-omy' presence,'without' presence (in Technique),'no' presence (in Technique),'without' before 'IV contrast'-like terms (in Technique),prediction
140,492361,2017-02-07 00:00:00.000,1. No acute abnormality within the abdomen and...,"eval ulcerative colitis flair, abscess",Following the administration of 150 cc of Omn...,True,True,True,False,True,...,False,False,False,False,False,False,False,False,False,Positive
145,492361,2020-07-17 00:00:00.000,History of proctocolectomy and left lower quad...,"evaluate for colitis please, 1 week of water...",Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,Positive
151,546011,2017-06-23 00:00:00.000,1. No acute intra-abdominal process. Interval...,Prior CTs showed hepatic flexure mass. Subseq...,Axial CT images were performed from the lung ...,True,False,False,False,True,...,False,False,False,False,False,False,True,False,False,Positive
249,1073222,2013-02-24 00:00:00.000,IMPRESSION: Interval decrease in proximal smal...,"pt with h/o colitis, possible SBO, but now wi...",Following the administration of 130 cc of Omn...,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,Positive
253,1087533,2019-08-03 00:00:00.000,1. Colonic wall thickening and mucosal hypere...,"hx ulcerative colitis, diverticulitis",Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,Positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13423,69944345,2020-02-26 00:00:00.000,"1. Mild ascending colon inflammatory changes, ...","eval pump position, any major colitis/abscess...",Contiguous 1.25-mm collimation axial images w...,True,False,False,False,True,...,False,False,False,False,False,False,True,False,True,Positive
13428,69944345,2020-05-16 00:00:00.000,Interval improvement of colitis involving the ...,colitis,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,Positive
13440,73193702,2020-04-28 00:00:00.000,CT findings compatible with ulcerative colitis...,Eval for intraabdominal infection and UC flar...,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,True,False,False,False,False,False,False,False,False,Positive
13483,76713138,2020-05-28 00:00:00.000,CT ENTEROGRAPHY 5/28/2020 CLINICAL HISTORY: ...,"24M w/ UC p/w acute UC flare w/ anemia, on re...",Following administration of oral Volumen and...,True,False,False,False,True,...,True,False,False,False,False,False,False,False,False,Positive


In [23]:
post_2012_copy_Crohn = post_2012.copy()
def pred_Crohn(df):
    neg = df["'ulcerative' presence"] + df["'colitis' presence"] + df["'proctitis' presence"] + df["'\sUC\s' presence"] + df["'rectosigmoiditis' presence"] > 0
    pos = df["'Crohn' presence"] + df["'Chron' presence"] + df["'proc-omy' presence"] + df["'ileo-omy' presence"] > 0
    
    conditions = [
    (neg),
    (pos) 
    ]

    # create a list of the values we want to assign for each condition
    values = ['Negative', 'Positive']

    # create a new column and use np.select to assign values to it using our lists as arguments
    df['prediction'] = np.select(conditions, values)
    return df
post_2012_copy_Crohn = pred_Crohn(post_2012_copy_Crohn)
post_2012_copy_Crohn.to_csv('post_2012_copy_Crohn.csv')

In [55]:
import medspacy

In [60]:
import spacy
import negspacy
from negspacy import Negex

ImportError: cannot import name 'Negex'

In [25]:
nlp = spacy.load('en_core_web_sm')
# Process whole documents
text = ("When Sebastian Thrun started working on self-driving cars at "
        "Google in 2007, few people outside of the company took him "
        "seriously. “I can tell you very senior CEOs of major American "
        "car companies would shake my hand and turn away because I wasn’t "
        "worth talking to,” said Thrun, in an interview with Recode earlier "
        "this week.")
doc = nlp(text)

# Analyze syntax
print("Noun phrases:", [chunk.text for chunk in doc.noun_chunks])
print("Verbs:", [token.lemma_ for token in doc if token.pos_ == "VERB"])

# Find named entities, phrases and concepts
for entity in doc.ents:
    print(entity.text, entity.label_)

Noun phrases: ['Sebastian Thrun', 'self-driving cars', 'Google', 'few people', 'the company', 'him', 'I', 'you', 'very senior CEOs', 'major American car companies', 'my hand', 'I', 'Thrun', 'an interview', 'Recode']
Verbs: ['start', 'work', 'drive', 'take', 'can', 'tell', 'would', 'shake', 'turn', 'talk', 'say']
Sebastian NORP
Google ORG
2007 DATE
American NORP
Recode ORG
earlier this week DATE


In [26]:
clinical_note = "Patient resting in bed. Patient given azithromycin without any difficulty. Patient has audible wheezing, \
states chest tightness. No evidence of hypertension.\
Patient denies nausea at this time. zofran declined. Patient is also having intermittent sweating associated with pneumonia. \
Patient refused pain but tylenol still given. Neither substance abuse nor alcohol use however cocaine once used in the last year. Alcoholism unlikely.\
Patient has headache and fever. Patient is not diabetic. \
No signs of diarrhea. Lab reports confirm lymphocytopenia. Cardaic rhythm is Sinus bradycardia. \
Patient also has a history of cardiac injury. No kidney injury reported. No abnormal rashes or ulcers. \
Patient might not have liver disease. Confirmed absence of hemoptysis. Although patient has severe pneumonia and fever \
, test reports are negative for COVID-19 infection. COVID-19 viral infection absent."
#lemmatizing the notes to capture all forms of negation(e.g., deny: denies, denying)
def lemmatize(note, nlp):
    doc = nlp(note)
    lemNote = [wd.lemma_ for wd in doc]
    return " ".join(lemNote)
lem_clinical_note= lemmatize(clinical_note, nlp)
#creating a doc object using BC5CDR model
doc = nlp(lem_clinical_note)


In [59]:
# nlp = medspacy.load("en_core_web_sm")
# for x in post_2012_copy_Crohn['impressions'][:10]:
#     doc = nlp(x)
#     visualize_ent(doc)

In [61]:
#adding a new pipeline component to identify negation
def neg_model(nlp_model):
    nlp = spacy.load(nlp_model, disable = ['parser'])
    nlp.add_pipe(nlp.create_pipe('sentencizer'))
    negex = Negex(nlp)
    nlp.add_pipe(negex)
    return nlp
"""
Negspacy sets a new attribute e._.negex to True if a negative concept is encountered
"""
def negation_handling(nlp_model, note, neg_model):
    results = []
    nlp = neg_model(nlp_model) 
    note = note.split(".") #sentence tokenizing based on delimeter 
    note = [n.strip() for n in note] #removing extra spaces at the begining and end of sentence
    for t in note:
        doc = nlp(t)
        for e in doc.ents:
            rs = str(e._.negex)
            if rs == "True": 
                results.append(e.text)
    return results
# #list of negative concepts from clinical note identified by negspacy
results0 = negation_handling("en_core_web_sm", lem_clinical_note, neg_model)

NameError: name 'Negex' is not defined

In [28]:
# ivc_df = indic_imp(post_2012)
# ivc_df = indic_num(post_2012)
# ivc_df =presence_multiple(ivc_df)

In [29]:
#len(ivc_df["'IV Contrast' presence"] == True)


In [30]:
ivc_filter= post_2012.copy()
ivc_filter = ivc_filter[ivc_filter["'without' before 'IV contrast'-like terms (in Technique)"] == False]


In [31]:
ivc_filter = pred_indic(ivc_filter)
ivc_filter = ivc_filter[ivc_filter['prediction'] == 'Positive']
ivc_filter

/opt/ars-virtenv/lib/python3.6/site-packages/pandas/core/computation/expressions.py:194: UserWarning: evaluating in Python space because the '+' operator is not supported by numexpr for the bool dtype, use '|' instead
  op=op_str, alt_op=unsupported[op_str]


,mrn,service_date,result,Clinical Indicators (Combined),technique indicators,'IV Contrast' presence,'oral contrast' presence,'rectal contrast' presence,'MRI' presence,CT,...,'\sUC\s' presence,'Crohn' presence,'Chron' presence,'rectosigmoiditis' presence,'proc-omy' presence,'ileo-omy' presence,'without' presence (in Technique),'no' presence (in Technique),'without' before 'IV contrast'-like terms (in Technique),prediction
140,492361,2017-02-07 00:00:00.000,1. No acute abnormality within the abdomen and...,"eval ulcerative colitis flair, abscess",Following the administration of 150 cc of Omn...,True,True,True,False,True,...,False,False,False,False,False,False,False,False,False,Positive
145,492361,2020-07-17 00:00:00.000,History of proctocolectomy and left lower quad...,"evaluate for colitis please, 1 week of water...",Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,Positive
151,546011,2017-06-23 00:00:00.000,1. No acute intra-abdominal process. Interval...,Prior CTs showed hepatic flexure mass. Subseq...,Axial CT images were performed from the lung ...,True,False,False,False,True,...,False,False,False,False,False,False,True,False,False,Positive
249,1073222,2013-02-24 00:00:00.000,IMPRESSION: Interval decrease in proximal smal...,"pt with h/o colitis, possible SBO, but now wi...",Following the administration of 130 cc of Omn...,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,Positive
253,1087533,2019-08-03 00:00:00.000,1. Colonic wall thickening and mucosal hypere...,"hx ulcerative colitis, diverticulitis",Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,Positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13400,69551238,2020-05-06 00:00:00.000,1. Interval surgical laparotomy with drainage ...,24 y/O F with UC elevated WBC concern for in...,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,True,False,False,False,False,False,False,False,False,Positive
13428,69944345,2020-05-16 00:00:00.000,Interval improvement of colitis involving the ...,colitis,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,Positive
13440,73193702,2020-04-28 00:00:00.000,CT findings compatible with ulcerative colitis...,Eval for intraabdominal infection and UC flar...,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,True,False,False,False,False,False,False,False,False,Positive
13483,76713138,2020-05-28 00:00:00.000,CT ENTEROGRAPHY 5/28/2020 CLINICAL HISTORY: ...,"24M w/ UC p/w acute UC flare w/ anemia, on re...",Following administration of oral Volumen and...,True,False,False,False,True,...,True,False,False,False,False,False,False,False,False,Positive


In [32]:
post_2012_copy_Crohn = post_2012_copy_Crohn[post_2012_copy_Crohn["'without' before 'IV contrast'-like terms (in Technique)"] == False]
post_2012_copy_Crohn = post_2012_copy_Crohn[post_2012_copy_Crohn['prediction'] == 'Positive']

In [33]:
ivc_filter['impressions1'] = ivc_filter['result'].str.extract(r'IMPRESSION:(.*) END OF IMPRESSION')
ivc_filter['impressions2'] = ivc_filter['result'].str.extract(r'1.\s(.*) END OF IMPRESSION')
ivc_filter['impressions3'] = ivc_filter['result'].str.extract(r'(.*) END OF IMPRESSION')
ivc_filter['impressions4'] = ivc_filter['result'].str.extract(r'IMPRESSION:(.*?):')
ivc_filter['impressions5'] = ivc_filter['result'].str.extract(r'1.\s(.*?):')
ivc_filter.to_csv('ivc_filter.csv')
ivc_filter
#ivc_t = break_apart_imps(ivc_filter)



,mrn,service_date,result,Clinical Indicators (Combined),technique indicators,'IV Contrast' presence,'oral contrast' presence,'rectal contrast' presence,'MRI' presence,CT,...,'ileo-omy' presence,'without' presence (in Technique),'no' presence (in Technique),'without' before 'IV contrast'-like terms (in Technique),prediction,impressions1,impressions2,impressions3,impressions4,impressions5
140,492361,2017-02-07 00:00:00.000,1. No acute abnormality within the abdomen and...,"eval ulcerative colitis flair, abscess",Following the administration of 150 cc of Omn...,True,True,True,False,True,...,False,False,False,False,Positive,NaN,No acute abnormality within the abdomen and pe...,1. No acute abnormality within the abdomen and...,CT ABDOMEN/PELVIS WITH CONTRAST 2/7/2017 1 C...,No acute abnormality within the abdomen and pe...
145,492361,2020-07-17 00:00:00.000,History of proctocolectomy and left lower quad...,"evaluate for colitis please, 1 week of water...",Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,TECHNIQUE
151,546011,2017-06-23 00:00:00.000,1. No acute intra-abdominal process. Interval...,Prior CTs showed hepatic flexure mass. Subseq...,Axial CT images were performed from the lung ...,True,False,False,False,True,...,False,True,False,False,Positive,NaN,NaN,NaN,NaN,No acute intra-abdominal process. Interval re...
249,1073222,2013-02-24 00:00:00.000,IMPRESSION: Interval decrease in proximal smal...,"pt with h/o colitis, possible SBO, but now wi...",Following the administration of 130 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,Interval decrease in proximal small bowel dis...,Appearance is suggestive of a closed loop obst...,IMPRESSION: Interval decrease in proximal smal...,Interval decrease in proximal small bowel dis...,Appearance is suggestive of a closed loop obst...
253,1087533,2019-08-03 00:00:00.000,1. Colonic wall thickening and mucosal hypere...,"hx ulcerative colitis, diverticulitis",Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,Colonic wall thickening and mucosal hyperenha...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13400,69551238,2020-05-06 00:00:00.000,1. Interval surgical laparotomy with drainage ...,24 y/O F with UC elevated WBC concern for in...,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,Interval surgical laparotomy with drainage of ...
13428,69944345,2020-05-16 00:00:00.000,Interval improvement of colitis involving the ...,colitis,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,NaN
13440,73193702,2020-04-28 00:00:00.000,CT findings compatible with ulcerative colitis...,Eval for intraabdominal infection and UC flar...,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,NaN
13483,76713138,2020-05-28 00:00:00.000,CT ENTEROGRAPHY 5/28/2020 CLINICAL HISTORY: ...,"24M w/ UC p/w acute UC flare w/ anemia, on re...",Following administration of oral Volumen and...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,NaN


In [34]:
post_2012_copy_Crohn['impressions1'] = post_2012_copy_Crohn['result'].str.extract(r'IMPRESSION:(.*) END OF IMPRESSION')
post_2012_copy_Crohn['impressions2'] = post_2012_copy_Crohn['result'].str.extract(r'1.\s(.*) END OF IMPRESSION')
post_2012_copy_Crohn['impressions3'] = post_2012_copy_Crohn['result'].str.extract(r'(.*) END OF IMPRESSION')
post_2012_copy_Crohn['impressions4'] = post_2012_copy_Crohn['result'].str.extract(r'IMPRESSION:(.*?):')
post_2012_copy_Crohn['impressions5'] = post_2012_copy_Crohn['result'].str.extract(r'1.\s(.*?):')
post_2012_copy_Crohn.to_csv('Crohn_filter.csv')
post_2012_copy_Crohn

,mrn,service_date,result,Clinical Indicators (Combined),technique indicators,'IV Contrast' presence,'oral contrast' presence,'rectal contrast' presence,'MRI' presence,CT,...,'ileo-omy' presence,'without' presence (in Technique),'no' presence (in Technique),'without' before 'IV contrast'-like terms (in Technique),prediction,impressions1,impressions2,impressions3,impressions4,impressions5
99,427713,2012-06-08 00:00:00.000,IMPRESSION: 1. Mild nonspecific thickening in ...,Worsening diarrhea. History of Crohn's diseas...,Following the administration of 150 cc of Omn...,True,False,False,True,False,...,False,False,False,False,Positive,1. Mild nonspecific thickening in the descend...,Mild nonspecific thickening in the descending ...,IMPRESSION: 1. Mild nonspecific thickening in ...,1. Mild nonspecific thickening in the descend...,Mild nonspecific thickening in the descending ...
121,478075,2020-03-13 00:00:00.000,History of Crohn's disease status post ileocol...,"47 yo F with Crohns dz, hx lung ca, w profoun...",Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,NaN
123,490539,2019-08-05 00:00:00.000,1. Distal proctocolitis. 2. 2.7 cm adnexal c...,"?SBO- diffuse abd pain, bloating, Decrease PO...",Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,Distal proctocolitis. 2. 2.7 cm adnexal cyst...
142,492361,2017-08-16 00:00:00.000,1. No acute intra-abdominal or intrapelvic fin...,"lower abdominal pain, s/p colectomy ADDITIONA...",Following the administration of 150 cc of Omn...,True,False,False,False,True,...,True,False,False,False,Positive,NaN,No acute intra-abdominal or intrapelvic findin...,1. No acute intra-abdominal or intrapelvic fin...,CT ABDOMEN/PELVIS WITH CONTRAST 8/16/2017 C...,No acute intra-abdominal or intrapelvic findin...
147,536202,2012-11-30 00:00:00.000,IMPRESSION: The distal ileum demonstrates thic...,CT enterography for Crohn's disease APEX ORDER,Following administration of oral Volumen and ...,True,False,False,False,True,...,False,False,True,False,Positive,The distal ileum demonstrates thickened hyper...,Patient was observed 45 minutes with no respir...,IMPRESSION: The distal ileum demonstrates thic...,The distal ileum demonstrates thickened hyper...,Patient was observed 45 minutes with no respir...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13498,77366318,2020-06-03 00:00:00.000,1. Interval increase in centrilobular nodules ...,POD#6 s/p Lap converted to open SBR ileostomy...,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,True,False,False,False,Positive,NaN,NaN,NaN,NaN,Interval increase in centrilobular nodules at ...
13499,77609202,2020-05-26 00:00:00.000,1. Marked lung segment wall thickening of the...,abd pain. Additional history: Crohn's disease...,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,Marked lung segment wall thickening of the di...
13500,77609202,2020-07-24 00:00:00.000,1. Persistent long segment wall thickening and...,c/f infection hx chrons and abscess. ADDITI...,Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,Persistent long segment wall thickening and ma...
13510,78167572,2020-07-28 00:00:00.000,1. Fecalization of intraluminal contents throu...,"ileal inflammation, no chronicity on biopsies",Following the administration of 150 cc of Omn...,True,False,False,False,True,...,False,False,False,False,Positive,NaN,NaN,NaN,NaN,Fecalization of intraluminal contents througho...


In [35]:
# post_2012_copy_Crohn['impressions'] = post_2012_copy_Crohn[post_2012_copy_Crohn.columns[-5:]].apply(
#     lambda x: ','.join(x.dropna().astype(str)),
#     axis=1
#)
# post_2012_copy_Crohn['impressions'].isna()

In [36]:
post_2012_copy_Crohn['impressions'] = post_2012_copy_Crohn['impressions1']
post_2012_copy_Crohn['impressions'].fillna(post_2012_copy_Crohn['impressions5'], inplace=True)
post_2012_copy_Crohn['impressions'].fillna(post_2012_copy_Crohn['impressions3'], inplace=True)
post_2012_copy_Crohn['impressions'].fillna(post_2012_copy_Crohn['impressions4'], inplace=True)
post_2012_copy_Crohn['impressions'].fillna(post_2012_copy_Crohn['impressions2'], inplace=True)

In [37]:
post_2012_copy_Crohn.to_csv('post_2012_copy_Crohn.csv')

In [38]:
# ivc_filter['impressions'] = np.where(ivc_filter['impressions1']==True, ivc_filter['impressions1'], 
#                                      np.where(ivc_filter['impressions5'] == True, ivc_filter['impressions5'],
#                                               np.where(ivc_filter['impressions3'] == True, ivc_filter['impressions3'], 
#                                                       np.where(ivc_filter['impressions4'] == True, ivc_filter['impressions4'], ivc_filter['impressions2']))))


In [39]:
# # create a list of our conditions
# conditions = [
#     (ivc_filter['impressions1'] == True),
#     (ivc_filter['impressions2'] == True & ivc_filter['impressions4'] != True), 
#     (ivc_filter['impressions3'] == True & ivc_filter['impressions5'] != True),
#     (ivc_filter['impressions4'] == True & ivc_filter['impressions3'] != True),
#     (ivc_filter['impressions5'] == True & ivc_filter['impressions1'] != True)
#     ]

# # create a list of the values we want to assign for each condition
# values = [ivc_filter['impressions1'], ivc_filter['impressions2'], ivc_filter['impressions3'], ivc_filter['impressions4'], ivc_filter['impressions5']]

# # create a new column and use np.select to assign values to it using our lists as arguments
# ivc_filter['impressions'] = np.select(conditions, values)

# ivc_filter

In [40]:

# for index, row in c.iterrows():
#     if index['impressions1'] == True:
#         c.loc[index, "impressions"] = row['impressions1']
#     elif row['impressions5'] == True:
#         c.loc[index,  row] = row['impressions5']
#     elif row['impressions3'] == True:
#         c.loc[index,  row] = row['impressions3']
#     elif row['impressions4'] == True:
#         c.loc[index,  row] = row['impressions4']
#     else:
#         c.loc[index,  row] = row['impressions2']
# c

In [41]:
def break_apart_imp(notes):
    temp = notes['impressions'].str.split(r'\d\.\s|(?<!Dr)\.')
#     for i in temp:
#         idx = 0
#         #temp[idx] = [j for j in temp[idx] if j != " "]
#         temp[idx] = [k for k in temp[idx]]
#         idx += 1
    combine = notes[["mrn", "service_date"]]
    return combine.merge(temp, how='outer', left_index=True, right_index=True)
#         for x in temp[idx]:
#             notes.append(notes.iloc[[idx]])
#             notes.loc[idx, ["impressions"]] = x
#         idx += 1
#     return notes
Crohn = post_2012_copy_Crohn.copy()
Crohn = break_apart_imp(Crohn)#.to_csv("test_imps_breakdown.csv")


In [42]:
Crohn_explode = Crohn[['mrn', 'service_date', 'impressions']].explode("impressions")
Crohn_explode

,mrn,service_date,impressions
99,427713,2012-06-08 00:00:00.000,
99,427713,2012-06-08 00:00:00.000,Mild nonspecific thickening in the descending ...
99,427713,2012-06-08 00:00:00.000,Small locule of gas seen along the medial wal...
99,427713,2012-06-08 00:00:00.000,"However, a small contained perforation is not..."
99,427713,2012-06-08 00:00:00.000,Correlation can be made with colonoscopy as c...
...,...,...,...
13510,78167572,2020-07-28 00:00:00.000,No visualized evidence of acute inflammation
13510,78167572,2020-07-28 00:00:00.000,
13510,78167572,2020-07-28 00:00:00.000,Heterogenous peripheral hepatic enhancement is...
13510,78167572,2020-07-28 00:00:00.000,Report dictated by


In [43]:
Crohn_explode["impressions"].replace('', np.nan, inplace=True)
Crohn_explode["impressions"].replace(' ', np.nan, inplace=True)
#total.replace(r'^\s*$', np.nan, regex=True)
Crohn_explode.dropna(inplace=True)
Crohn_explode = Crohn_explode[Crohn_explode["impressions"].str.len() > 1]
Crohn_explode

,mrn,service_date,impressions
99,427713,2012-06-08 00:00:00.000,Mild nonspecific thickening in the descending ...
99,427713,2012-06-08 00:00:00.000,Small locule of gas seen along the medial wal...
99,427713,2012-06-08 00:00:00.000,"However, a small contained perforation is not..."
99,427713,2012-06-08 00:00:00.000,Correlation can be made with colonoscopy as c...
99,427713,2012-06-08 00:00:00.000,Mild ascites
...,...,...,...
13500,77609202,2020-07-24 00:00:00.000,Report dictated by
13510,78167572,2020-07-28 00:00:00.000,Fecalization of intraluminal contents througho...
13510,78167572,2020-07-28 00:00:00.000,No visualized evidence of acute inflammation
13510,78167572,2020-07-28 00:00:00.000,Heterogenous peripheral hepatic enhancement is...


In [44]:
Crohn_explode.to_csv("exploded_Crohn.csv")

In [45]:
Crohn_explode["'inflamm*' presence"] = [bool(re.search(r"inflamm*", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'fat stranding' presence"] = [bool(re.search(r"fat stranding", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'fat stra*' presence"] = [bool(re.search(r"fat stra.*", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'fat*' presence"] = [bool(re.search(r"fat.*", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'acute' presence"] = [bool(re.search(r"acute", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'fistula' presence"] = [bool(re.search(r"fistula", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'enhancement' presence"] = [bool(re.search(r"enhancement", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'hyperenhancement' presence"] = [bool(re.search(r"hyperenhancement", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'hyperemia' presence"] = [bool(re.search(r"hyperemia", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'stranding' presence"] = [bool(re.search(r"stranding", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'chronic' presence"] = [bool(re.search(r"chronic", x)) for x in Crohn_explode["impressions"]]

Crohn_explode["'wall thickening' presence"] = [bool(re.search(r"wall thickening", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'wall thick*' presence"] = [bool(re.search(r"wall thick*", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'edema' presence"] = [bool(re.search(r"edema", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'hypere*' presence"] = [bool(re.search(r"hypere*", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'strand*' presence"] = [bool(re.search(r"strand*", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'colitis' presence"] = [bool(re.search(r"colitis", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'large lymph node' presence"] = [bool(re.search(r"large lymph node", x)) for x in Crohn_explode["impressions"]]

Crohn_explode["'thickening' presence"] = [bool(re.search(r"thickening", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'thicken*' presence"] = [bool(re.search(r"thicken.*", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'abscess' presence"] = [bool(re.search(r"abscess", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'segmental mural hyperenhancement' presence"] = [bool(re.search(r"segmental mural hyperenhancement", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'asymmetric wall thickening' presence"] = [bool(re.search(r"asymmetric wall thickening", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'ulcerations' presence"] = [bool(re.search(r"ulcerations", x)) for x in Crohn_explode["impressions"]] 
Crohn_explode["'sacculations' presence"] = [bool(re.search(r"sacculations", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'intramural edema' presence"] = [bool(re.search(r"intramural edema", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'stricture' presence"] = [bool(re.search(r"stricture", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'simple fistula' presence"] = [bool(re.search(r"simple fistula", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'complex fistula' presence"] = [bool(re.search(r"complex fistula", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'sinus tract' presence"] = [bool(re.search(r"sinus tract", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'perianal fistula' presence"] = [bool(re.search(r"perianal fistula", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'inflammatory mass' presence"] = [bool(re.search(r"inflammatory mass", x)) for x in Crohn_explode["impressions"]]

Crohn_explode["'perienteric edema/inflammation' presence"] = [bool(re.search(r"perienteric edema|perienteric inflammation", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'engorged vasa recta' presence"] = [bool(re.search(r"engorged vasa recta", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'fibrofatty proliferation' presence"] = [bool(re.search(r"fibrofatty proliferation", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'mesenteric venous thrombosis/ occlusion' presence"] = [bool(re.search(r"mesenteric venous thrombosis|mesenteric venous occlusion", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'adenopathy' presence"] = [bool(re.search(r"adenopathy", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'obstruction' presence"] = [bool(re.search(r"obstruction", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'free' presence"] = [bool(re.search(r"free", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'colitis' presence"] = [bool(re.search(r"colitis", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'evidence' presence"] = [bool(re.search(r"evidence", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'increased' presence"] = [bool(re.search(r"increased", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'crohn' presence"] = [bool(re.search(r"crohn", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'mucosal' presence"] = [bool(re.search(r"mucosal", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'infiltration' presence"] = [bool(re.search(r"infiltration", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'pericolonic' presence"] = [bool(re.search(r"pericolonic", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'active' presence"] = [bool(re.search(r"active", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'mild' presence"] = [bool(re.search(r"[Mm]ild", x)) for x in Crohn_explode["impressions"]]

#uncertainty
Crohn_explode["'no' presence (Negation)"] = [bool(re.search(r"[Nn]o\s", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'without' presence (Negation)"] = [bool(re.search(r"without", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'not' presence (Negation)"] = [bool(re.search(r"[Nn]ot\s", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'no' in same sentence as 'acute' presence"] = [bool(re.search(r"(?=.*\b[Nn]o\b)(?=.*\bacute\b)", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'no' in same sentence as 'chronic' presence"] = [bool(re.search(r"(?=.*\b[Nn]o\b)(?=.*\bchronic\b)", x) ) for x in Crohn_explode["impressions"]]
Crohn_explode["'suspected' presence"] = [bool(re.search(r"[Ss]uspected", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'query' presence"] = [bool(re.search(r"query", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'possible' presence"] = [bool(re.search(r"possible", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'rule out' presence"] = [bool(re.search(r"rule out", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'borderline' presence"] = [bool(re.search(r"borderline", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'indeterminate' presence"] = [bool(re.search(r"indeterminate", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'no definite' presence"] = [bool(re.search(r"no definite", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'probable' presence"] = [bool(re.search(r"probable", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'no obvious' presence"] = [bool(re.search(r"no obvious", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'equivocal' presence"] = [bool(re.search(r"equivocal", x)) for x in Crohn_explode["impressions"]]
Crohn_explode["'suggested|suggests' presence"] = [bool(re.search(r"suggested|suggests", x)) for x in Crohn_explode["impressions"]]

/opt/ars-virtenv/lib/python3.6/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/opt/ars-virtenv/lib/python3.6/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/opt/ars-virtenv/lib/python3.6/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

In [46]:
Crohn_explode.to_csv("Crohn_fulldf.csv")

SNORKEL

In [47]:

df_train = Crohn_explode.sample(frac=.5)


In [48]:
# Define the label mappings for convenience
ABSTAIN = -1
NEGATIVE = 0
POSITIVE = 1

In [49]:
from snorkel.labeling import labeling_function
#Labeling functions

@labeling_function()
def lf_keyword_abcess(x):
    return POSITIVE if "abcess" in x.text.lower() else ABSTAIN

In [50]:
def lf_regex(x):
    
    return POSITIVE if re.search(r"active", x.text, flags=re.I) else ABSTAIN

In [51]:
@labeling_function()
def lf_neg(x):
    """Non-spam comments are often short, such as 'cool video!'."""
    return NEGATIVE if re.search(r"\s[Nn]o\s", x.text, flags=re.I) else ABSTAIN

In [ ]:
# from snorkel.labeling.model import LabelModel
# from snorkel.labeling import PandasLFApplier

# # Define the set of labeling functions (LFs)
# lfs = [lf_keyword_abcess, lf_regex, lf_neg]

# # Apply the LFs to the unlabeled training data
# applier = PandasLFApplier(lfs)
# L_train = applier.apply(df_train)

# # Train the label model and compute the training labels
# label_model = LabelModel(cardinality=2, verbose=True)
# label_model.fit(L_train, n_epochs=500, log_freq=50, seed=123)
# df_train["label"] = label_model.predict(L=L_train, tie_break_policy="abstain")

In [ ]:
df_train = df_train[df_train.label != ABSTAIN]

NameError: name 'POSITIVE' is not defined

In [52]:
def pred_full(df):
    p = df.iloc[:, 3:49]
    n = df.iloc[:, 49:]
    conditions = [
     df["'mild' presence"]+ df["'no' presence (Negation)"] +df["'without' presence (Negation)"] + df["'not' presence (Negation)"]+ df["'no' in same sentence as 'acute' presence"]+ df["'no' in same sentence as 'chronic' presence"]+ df["'suspected' presence"]+ df["'query' presence"]+df["'possible' presence"]+ df["'rule out' presence"]+ df["'borderline' presence"]+ df["'indeterminate' presence"]+ df["'no definite' presence"]+df["'probable' presence"]+ df["'no obvious' presence"]+ df["'equivocal' presence"]+ df["'suggested|suggests' presence"]>0,
        df["'inflamm*' presence"]+ df["'fat stranding' presence"] + df["'fat stra*' presence"] + df["'fat*' presence"]+ df["'acute' presence"]+ df["'fistula' presence"]+	 df["'enhancement' presence"]+
        df["'hyperenhancement' presence"]+ df["'hyperemia' presence"]+ df["'stranding' presence"]+ df["'chronic' presence"] +df["'wall thickening' presence"]+ df["'wall thick*' presence"]+df["'edema' presence"]	+ df["'hypere*' presence"]+	 df["'strand*' presence"]+ df["'colitis' presence"]+ df["'large lymph node' presence"]+ df["'thickening' presence"]+
    df["'thicken*' presence"]+df["'abscess' presence"]+ df["'segmental mural hyperenhancement' presence"]+ df["'asymmetric wall thickening' presence"]+	 df["'ulcerations' presence"]+ df["'sacculations' presence"]+	
        df["'intramural edema' presence"]+df["'stricture' presence"]+df["'simple fistula' presence"] + df["'complex fistula' presence"]+df["'sinus tract' presence"]+ df["'perianal fistula' presence"]+ df["'inflammatory mass' presence"]+ df["'perienteric edema/inflammation' presence"]+df["'engorged vasa recta' presence"]+
        df["'fibrofatty proliferation' presence"]+ df["'mesenteric venous thrombosis/ occlusion' presence"]+ df["'adenopathy' presence"]+ df["'obstruction' presence"] + df["'free' presence"]+ df["'evidence' presence"]+ df["'increased' presence"]+ df["'crohn' presence"]+ df["'mucosal' presence"]+ df["'infiltration' presence"]+ df["'pericolonic' presence"]+ df["'active' presence"] > 0
        						
    ]

    # create a list of the values we want to assign for each condition
    values = ['Negative', 'Positive']

    # create a new column and use np.select to assign values to it using our lists as arguments
    df['prediction'] = np.select(conditions, values)
    return df

predictions_Crohn = pred_full(Crohn_explode)
predictions_Crohn


/opt/ars-virtenv/lib/python3.6/site-packages/ipykernel_launcher.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


,mrn,service_date,impressions,'inflamm*' presence,'fat stranding' presence,'fat stra*' presence,'fat*' presence,'acute' presence,'fistula' presence,'enhancement' presence,...,'possible' presence,'rule out' presence,'borderline' presence,'indeterminate' presence,'no definite' presence,'probable' presence,'no obvious' presence,'equivocal' presence,'suggested|suggests' presence,prediction
99,427713,2012-06-08 00:00:00.000,Mild nonspecific thickening in the descending ...,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,Negative
99,427713,2012-06-08 00:00:00.000,Small locule of gas seen along the medial wal...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0
99,427713,2012-06-08 00:00:00.000,"However, a small contained perforation is not...",False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,Negative
99,427713,2012-06-08 00:00:00.000,Correlation can be made with colonoscopy as c...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0
99,427713,2012-06-08 00:00:00.000,Mild ascites,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,Negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13500,77609202,2020-07-24 00:00:00.000,Report dictated by,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0
13510,78167572,2020-07-28 00:00:00.000,Fecalization of intraluminal contents througho...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0
13510,78167572,2020-07-28 00:00:00.000,No visualized evidence of acute inflammation,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,Negative
13510,78167572,2020-07-28 00:00:00.000,Heterogenous peripheral hepatic enhancement is...,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,Positive


In [53]:
predictions_Crohn.to_csv('predictions_Crohn.csv')

OSError: [Errno 16] Device or resource busy: 'predictions_Crohn.csv'

In [ ]:
ivc_filter.to_csv('ivc_filter_newimps.csv')

In [ ]:
ivc_filter

In [ ]:
iv_break = ivc_filter.copy()
iv_break['impressions'] = iv_break['impressions1']
iv_break['impressions'].fillna(iv_break['impressions5'], inplace=True)
iv_break['impressions'].fillna(iv_break['impressions3'], inplace=True)
iv_break['impressions'].fillna(iv_break['impressions4'], inplace=True)
iv_break['impressions'].fillna(iv_break['impressions2'], inplace=True)
iv_break = break_apart_imp(iv_break)#.to_csv("test_imps_breakdown.csv")


In [ ]:
iv_break

In [ ]:
iv_explode = iv_break[['mrn', 'service_date', 'impressions']].explode("impressions")
iv_explode


In [ ]:
iv_explode["impressions"].replace('', np.nan, inplace=True)

iv_explode.replace(r'^\s*$', np.nan, regex=True)
iv_explode.dropna(inplace=True)
iv_explode = iv_explode[iv_explode["impressions"].str.len() >= 1]
iv_explode

In [ ]:
iv_explode["'inflamm*' presence"] = [bool(re.search(r"inflamm*", x)) for x in iv_explode["impressions"]]
iv_explode["'fat stranding' presence"] = [bool(re.search(r"fat stranding", x)) for x in iv_explode["impressions"]]
iv_explode["'fat stra*' presence"] = [bool(re.search(r"fat stra.*", x)) for x in iv_explode["impressions"]]
iv_explode["'fat*' presence"] = [bool(re.search(r"fat.*", x)) for x in iv_explode["impressions"]]
iv_explode["'acute' presence"] = [bool(re.search(r"acute", x)) for x in iv_explode["impressions"]]
iv_explode["'fistula' presence"] = [bool(re.search(r"fistula", x)) for x in iv_explode["impressions"]]
iv_explode["'enhancement' presence"] = [bool(re.search(r"enhancement", x)) for x in iv_explode["impressions"]]
iv_explode["'hyperenhancement' presence"] = [bool(re.search(r"hyperenhancement", x)) for x in iv_explode["impressions"]]
iv_explode["'hyperemia' presence"] = [bool(re.search(r"hyperemia", x)) for x in iv_explode["impressions"]]
iv_explode["'stranding' presence"] = [bool(re.search(r"stranding", x)) for x in iv_explode["impressions"]]
iv_explode["'chronic' presence"] = [bool(re.search(r"chronic", x)) for x in iv_explode["impressions"]]

iv_explode["'wall thickening' presence"] = [bool(re.search(r"wall thickening", x)) for x in iv_explode["impressions"]]
iv_explode["'wall thick*' presence"] = [bool(re.search(r"wall thick*", x)) for x in iv_explode["impressions"]]
iv_explode["'edema' presence"] = [bool(re.search(r"edema", x)) for x in iv_explode["impressions"]]
iv_explode["'hypere*' presence"] = [bool(re.search(r"hypere*", x)) for x in iv_explode["impressions"]]
iv_explode["'strand*' presence"] = [bool(re.search(r"strand*", x)) for x in iv_explode["impressions"]]
iv_explode["'colitis' presence"] = [bool(re.search(r"colitis", x)) for x in iv_explode["impressions"]]
iv_explode["'large lymph node' presence"] = [bool(re.search(r"large lymph node", x)) for x in iv_explode["impressions"]]

iv_explode["'thickening' presence"] = [bool(re.search(r"thickening", x)) for x in iv_explode["impressions"]]
iv_explode["'thicken*' presence"] = [bool(re.search(r"thicken.*", x)) for x in iv_explode["impressions"]]
iv_explode["'abscess' presence"] = [bool(re.search(r"abscess", x)) for x in iv_explode["impressions"]]
iv_explode["'segmental mural hyperenhancement' presence"] = [bool(re.search(r"segmental mural hyperenhancement", x)) for x in iv_explode["impressions"]]
iv_explode["'asymmetric wall thickening' presence"] = [bool(re.search(r"asymmetric wall thickening", x)) for x in iv_explode["impressions"]]
iv_explode["'ulcerations' presence"] = [bool(re.search(r"ulcerations", x)) for x in iv_explode["impressions"]] 
iv_explode["'sacculations' presence"] = [bool(re.search(r"sacculations", x)) for x in iv_explode["impressions"]]
iv_explode["'intramural edema' presence"] = [bool(re.search(r"intramural edema", x)) for x in iv_explode["impressions"]]
iv_explode["'stricture' presence"] = [bool(re.search(r"stricture", x)) for x in iv_explode["impressions"]]
iv_explode["'simple fistula' presence"] = [bool(re.search(r"simple fistula", x)) for x in iv_explode["impressions"]]
iv_explode["'complex fistula' presence"] = [bool(re.search(r"complex fistula", x)) for x in iv_explode["impressions"]]
iv_explode["'sinus tract' presence"] = [bool(re.search(r"sinus tract", x)) for x in iv_explode["impressions"]]
iv_explode["'perianal fistula' presence"] = [bool(re.search(r"perianal fistula", x)) for x in iv_explode["impressions"]]
iv_explode["'inflammatory mass' presence"] = [bool(re.search(r"inflammatory mass", x)) for x in iv_explode["impressions"]]

iv_explode["'perienteric edema/inflammation' presence"] = [bool(re.search(r"perienteric edema|perienteric inflammation", x)) for x in iv_explode["impressions"]]
iv_explode["'engorged vasa recta' presence"] = [bool(re.search(r"engorged vasa recta", x)) for x in iv_explode["impressions"]]
iv_explode["'fibrofatty proliferation' presence"] = [bool(re.search(r"fibrofatty proliferation", x)) for x in iv_explode["impressions"]]
iv_explode["'mesenteric venous thrombosis/ occlusion' presence"] = [bool(re.search(r"mesenteric venous thrombosis|mesenteric venous occlusion", x)) for x in iv_explode["impressions"]]
iv_explode["'adenopathy' presence"] = [bool(re.search(r"adenopathy", x)) for x in iv_explode["impressions"]]
iv_explode["'obstruction' presence"] = [bool(re.search(r"obstruction", x)) for x in iv_explode["impressions"]]
iv_explode["'free' presence"] = [bool(re.search(r"free", x)) for x in iv_explode["impressions"]]
iv_explode["'colitis' presence"] = [bool(re.search(r"colitis", x)) for x in iv_explode["impressions"]]
iv_explode["'evidence' presence"] = [bool(re.search(r"evidence", x)) for x in iv_explode["impressions"]]
iv_explode["'increased' presence"] = [bool(re.search(r"increased", x)) for x in iv_explode["impressions"]]
iv_explode["'crohn' presence"] = [bool(re.search(r"crohn", x)) for x in iv_explode["impressions"]]
iv_explode["'mucosal' presence"] = [bool(re.search(r"mucosal", x)) for x in iv_explode["impressions"]]
iv_explode["'infiltration' presence"] = [bool(re.search(r"infiltration", x)) for x in iv_explode["impressions"]]
iv_explode["'pericolonic' presence"] = [bool(re.search(r"pericolonic", x)) for x in iv_explode["impressions"]]
iv_explode["'active' presence"] = [bool(re.search(r"active", x)) for x in iv_explode["impressions"]]
iv_explode["'mild' presence"] = [bool(re.search(r"mild", x)) for x in iv_explode["impressions"]]

#uncertainty
iv_explode["'no' presence (Negation)"] = [bool(re.search(r"\sno\s", x)) for x in iv_explode["impressions"]]
iv_explode["'without' presence (Negation)"] = [bool(re.search(r"without", x)) for x in iv_explode["impressions"]]
iv_explode["'not' presence (Negation)"] = [bool(re.search(r"\snot\s", x)) for x in iv_explode["impressions"]]
iv_explode["'no' in same sentence as 'acute' presence"] = [bool(re.search(r"(?=.*\bno\b)(?=.*\bacute\b)", x)) for x in iv_explode["impressions"]]
iv_explode["'no' in same sentence as 'chronic' presence"] = [bool(re.search(r"(?=.*\bno\b)(?=.*\bchronic\b)", x) ) for x in iv_explode["impressions"]]
iv_explode["'suspected' presence"] = [bool(re.search(r"suspected", x)) for x in iv_explode["impressions"]]
iv_explode["'query' presence"] = [bool(re.search(r"query", x)) for x in iv_explode["impressions"]]
iv_explode["'possible' presence"] = [bool(re.search(r"possible", x)) for x in iv_explode["impressions"]]
iv_explode["'rule out' presence"] = [bool(re.search(r"rule out", x)) for x in iv_explode["impressions"]]
iv_explode["'borderline' presence"] = [bool(re.search(r"borderline", x)) for x in iv_explode["impressions"]]
iv_explode["'indeterminate' presence"] = [bool(re.search(r"indeterminate", x)) for x in iv_explode["impressions"]]
iv_explode["'no definite' presence"] = [bool(re.search(r"no definite", x)) for x in iv_explode["impressions"]]
iv_explode["'probable' presence"] = [bool(re.search(r"probable", x)) for x in iv_explode["impressions"]]
iv_explode["'no obvious' presence"] = [bool(re.search(r"no obvious", x)) for x in iv_explode["impressions"]]
iv_explode["'equivocal' presence"] = [bool(re.search(r"equivocal", x)) for x in iv_explode["impressions"]]
iv_explode["'suggested|suggests' presence"] = [bool(re.search(r"suggested|suggests", x)) for x in iv_explode["impressions"]]

In [ ]:
iv_explode.to_csv('iv_fulldf.csv')

In [ ]:
iv_explode.iloc[:, 3:49]

In [ ]:
iv_explode.iloc[:, 49:]

In [ ]:
# d = iv_explode.iloc[:, 49:].columns
# a = list(d)
# iv_explode[[a]]

In [ ]:
def pred_full(df):
    p = df.iloc[:, 3:49]
    n = df.iloc[:, 49:]
    conditions = [
     df["'mild' presence"]+ df["'no' presence (Negation)"]+ df["'without' presence (Negation)"] +df["'not' presence (Negation)"]+ df["'no' in same sentence as 'acute' presence"]+ df["'no' in same sentence as 'chronic' presence"]+ df["'suspected' presence"]+ df["'query' presence"]+df["'possible' presence"]+ df["'rule out' presence"]+ df["'borderline' presence"]+ df["'indeterminate' presence"]+ df["'no definite' presence"]+df["'probable' presence"]+ df["'no obvious' presence"]+ df["'equivocal' presence"]+ df["'suggested|suggests' presence"]>0,
        df["'inflamm*' presence"]+ df["'fat stranding' presence"] + df["'fat stra*' presence"] + df["'fat*' presence"]+ df["'acute' presence"]+ df["'fistula' presence"]+	 df["'enhancement' presence"]+
        df["'hyperenhancement' presence"]+ df["'hyperemia' presence"]+ df["'stranding' presence"]+ df["'chronic' presence"] +df["'wall thickening' presence"]+ df["'wall thick*' presence"]+df["'edema' presence"]	+ df["'hypere*' presence"]+	 df["'strand*' presence"]+ df["'colitis' presence"]+ df["'large lymph node' presence"]+ df["'thickening' presence"]+
    df["'thicken*' presence"]+df["'abscess' presence"]+ df["'segmental mural hyperenhancement' presence"]+ df["'asymmetric wall thickening' presence"]+	 df["'ulcerations' presence"]+ df["'sacculations' presence"]+	
        df["'intramural edema' presence"]+df["'stricture' presence"]+df["'simple fistula' presence"] + df["'complex fistula' presence"]+df["'sinus tract' presence"]+ df["'perianal fistula' presence"]+ df["'inflammatory mass' presence"]+ df["'perienteric edema/inflammation' presence"]+df["'engorged vasa recta' presence"]+
        df["'fibrofatty proliferation' presence"]+ df["'mesenteric venous thrombosis/ occlusion' presence"]+ df["'adenopathy' presence"]+ df["'obstruction' presence"] + df["'free' presence"]+ df["'evidence' presence"]+ df["'increased' presence"]+ df["'crohn' presence"]+ df["'mucosal' presence"]+ df["'infiltration' presence"]+ df["'pericolonic' presence"]+ df["'active' presence"] > 0
        						
    ]

    # create a list of the values we want to assign for each condition
    values = ['Negative', 'Positive']

    # create a new column and use np.select to assign values to it using our lists as arguments
    df['prediction'] = np.select(conditions, values)
    return df
predictions_iv = pred_full(iv_explode)
predictions_iv
predictions_iv.to_csv('predictions_iv_UC.csv')

In [ ]:
# #to do:
# def pred_full(df):
#     p = df.iloc[:, 3:49]
#     pos = df[p.columns] 
#     n = df.iloc[:, 49:]
#     neg = df[n.columns]
#     conditions = [
#     (sum(pos) > 0),
#     (sum(neg) > 0)
#     ]

#     # create a list of the values we want to assign for each condition
#     values = ['Negative', 'Positive']

#     # create a new column and use np.select to assign values to it using our lists as arguments
#     df['prediction'] = np.select(conditions, values)
#     return df
# predictions_iv = pred_full(iv_explode)
# predictions_iv
# #predictions_iv.to_csv('predictions_iv.csv')



Initial Impressions Exploratory Work

In [ ]:
#Create indicator variable for Impression/End of Impression
def indic_imp(note):
    note["'impression' presence"] = note["result"].str.contains('(?<!OF\s)IMPRESSION:', regex=True)
    note["'end of impression' presence"] = note["result"].str.contains('END OF IMPRESSION', regex=True)
    return note
def indic_num(note):
    note["'1. \w' presence"] = note["result"].str.contains('^1\.\s\w+|\s1\.\s\w+', regex=True)
    note["'2. \w' presence"] = note["result"].str.contains('^2\.\s\w+|\s2\.\s\w+', regex=True)
    note["'3. \w' presence"] = note["result"].str.contains('^3\.\s\w+|\s3\.\s\w+', regex=True)
    return note
with_imps = indic_imp(initial_copy)
with_num = indic_num(initial_copy)
# with_num = with_num[with_num["'1. \w' presence"] == True]
# with_imps = with_imps[with_imps["'impression' presence"] == True]
# with_imps = with_imps[with_imps["'end of impression' presence"] == True]
initial_copy
                                                            
    
# def impression(note):
#     imps = re.findall(r'(IMPRESSION:.+|1\..+)Note|END OF IMPRESSION|\//', note)
#     if len(imps) > 0: 
#         cols = re.split(r'\.', imps[0])
#         cols = cols[1:len(cols)-1]
#         return make_columns(cols)
#     return
# initial_copy['result'].apply(impression)
# initial_copy.head()
#print(impression("IMPRESSION: . Dilated proximal and decompressed distal small bowel loops, consistent with small bowel obstruction. Small ascites. . Incompletely visualized lingular consolidation, may represent pneumonia. Note: These findings were discussed with Dr. Schnorr (ED) by Dr. Mutch (Radiology) at  AM on //. END OF IMPRESSION: CT ABDOMEN PEL W/O CONTRAST //  CLINICAL HISTORY: post renal transplant pain COMPARISON: Outside facility CT abdomen and pelvis"))

In [ ]:
with_imps.to_csv("Processed_with_imps.csv")

In [ ]:
def presence_multiple(note):
    #df = pd.DataFrame()
    for index, row in note.iterrows():
        count = row["'impression' presence"] + row["'end of impression' presence"] + row["'1. \w' presence"] + row["'2. \w' presence"]
        if count < 1:
            note.drop(index)
    #return df
    return note
presence_multiple(with_imps)
#filtered.to_csv("expanded_filtered.csv")
#with_num = indic_num(initial_copy)
# with_num = with_num[with_num["'1. \w' presence"] == True]
# with_num

In [18]:
#Create indicator variable for End of Impression
# def indic_end_imp(note):
#     note["'end of impression' presence"] = note["result"].str.contains('END OF IMPRESSION', regex=True)
#     return note
# with_both_imps = indic_end_imp(with_imps)
# with_both_imps = with_both_imps[with_both_imps["'end of impression' presence"] == True]

# with_both_imps.to_csv("Processed_with_both_imps.csv")
                                                          

In [19]:
filtered2 = with_num.copy()
filtered2['impressions'] = filtered2['result'].str.extract(r'IMPRESSION:(.*) END OF IMPRESSION')
filtered2

,mrn,service_date,result,'impression' presence,'end of impression' presence,'1. \w' presence,'2. \w' presence,'3. \w' presence,impressions
0,12181,2013-08-18 00:00:00.000,IMPRESSION: 1. Dilated proximal and decompress...,True,True,True,True,False,1. Dilated proximal and decompressed distal s...
1,18163,2013-08-20 00:00:00.000,STATUS: FINAL EXAM: CT ABD/PELVIS W/CONTRAST ...,False,False,False,False,False,NaN
2,68012,2000-11-13 00:00:00.000,Visit # XRAY DATE: 11-13-00 ...,True,False,True,True,False,NaN
3,68012,2007-12-21 00:00:00.000,ABDOMINAL PAIN CT OF THE ABDOMEN AND PELVIS: ...,True,True,False,False,False,No CT findings to explain the patient's histo...
4,68144,2018-04-04 00:00:00.000,1. Stool-filled colon. Marked rectal distenti...,False,False,False,False,False,NaN
...,...,...,...,...,...,...,...,...,...
13509,78119306,2020-06-05 00:00:00.000,1. Status post partial colectomy with left low...,False,False,True,True,False,NaN
13510,78167572,2020-07-28 00:00:00.000,1. Fecalization of intraluminal contents throu...,False,False,True,True,False,NaN
13511,78174358,2020-07-24 00:00:00.000,CT ENTEROGRAPHY 7/24/2020 CLINICAL HISTORY: ...,False,False,False,False,False,NaN
13512,78473037,2020-06-28 00:00:00.000,1. . Appendix dilation up to 10 mm with mucosa...,False,False,False,False,False,NaN


In [80]:
copy = with_imps.copy()
copy['impressions'] = copy['result'].str.extract(r'IMPRESSION:(.*) END OF IMPRESSION')
copy.to_csv("Processed_Extracted.csv")


NameError: name 'with_imps' is not defined

In [79]:
def break_apart_imps(notes):
    temp = notes['impressions'].str.split(r'\d\.\s|(?<!Dr)\.')
    for i in temp:
        idx = 0
        #temp[idx] = [j for j in temp[idx] if j != " "]
        temp[idx] = [k for k in temp[idx] if k != ""]
        idx += 1
    combine = notes[["mrn", "service_date"]]
    return combine.merge(temp, how='outer', left_index=True, right_index=True)
#         for x in temp[idx]:
#             notes.append(notes.iloc[[idx]])
#             notes.loc[idx, ["impressions"]] = x
#         idx += 1
#     return notes
total = break_apart_imps(copy)#.to_csv("test_imps_breakdown.csv")
total

NameError: name 'copy' is not defined

In [23]:
total.to_csv('imps.csv')

In [24]:
total = total.explode("impressions")
total
total["impressions"].replace('', np.nan, inplace=True)


In [25]:
total["impressions"].replace('', np.nan, inplace=True)
#total.replace(r'^\s*$', np.nan, regex=True)
total.dropna(inplace=True)
total = total[total["impressions"].str.len() > 1]
total

,mrn,service_date,impressions
0,12181,2013-08-18 00:00:00.000,Dilated proximal and decompressed distal small...
0,12181,2013-08-18 00:00:00.000,Small ascites
0,12181,2013-08-18 00:00:00.000,Incompletely visualized lingular consolidation...
0,12181,2013-08-18 00:00:00.000,Note: These findings were discussed with Dr. ...
3,68012,2007-12-21 00:00:00.000,No CT findings to explain the patient's histo...
...,...,...,...
13471,76384401,2013-01-24 00:00:00.000,No evidence of recurrence or metastatic disea...
13472,76384401,2013-06-03 00:00:00.000,No evidence for recurrence or metastatic disea...
13473,76384401,2013-11-21 00:00:00.000,No evidence for recurrence or metastatic dise...
13474,76384401,2014-05-12 00:00:00.000,No evidence of metastatic disease in the abdo...


In [26]:
total.to_csv('exploded_imps.csv')

In [27]:
total["'inflamm*' presence"] = [bool(re.search(r"inflamm*", x)) for x in total["impressions"]]
total["'fat stranding' presence"] = [bool(re.search(r"fat stranding", x)) for x in total["impressions"]]
total["'fat stra*' presence"] = [bool(re.search(r"fat stra.*", x)) for x in total["impressions"]]
total["'fat*' presence"] = [bool(re.search(r"fat.*", x)) for x in total["impressions"]]
total["'acute' presence"] = [bool(re.search(r"acute", x)) for x in total["impressions"]]
total["'fistula' presence"] = [bool(re.search(r"fistula", x)) for x in total["impressions"]]
total["'enhancement' presence"] = [bool(re.search(r"enhancement", x)) for x in total["impressions"]]
total["'hyperenhancement' presence"] = [bool(re.search(r"hyperenhancement", x)) for x in total["impressions"]]
total["'hyperemia' presence"] = [bool(re.search(r"hyperemia", x)) for x in total["impressions"]]
total["'stranding' presence"] = [bool(re.search(r"stranding", x)) for x in total["impressions"]]
total["'chronic' presence"] = [bool(re.search(r"chronic", x)) for x in total["impressions"]]

total["'wall thickening' presence"] = [bool(re.search(r"wall thickening", x)) for x in total["impressions"]]
total["'wall thick*' presence"] = [bool(re.search(r"wall thick*", x)) for x in total["impressions"]]
total["'edema' presence"] = [bool(re.search(r"edema", x)) for x in total["impressions"]]
total["'hypere*' presence"] = [bool(re.search(r"hypere*", x)) for x in total["impressions"]]
total["'strand*' presence"] = [bool(re.search(r"strand*", x)) for x in total["impressions"]]
total["'colitis' presence"] = [bool(re.search(r"colitis", x)) for x in total["impressions"]]
total["'large lymph node' presence"] = [bool(re.search(r"large lymph node", x)) for x in total["impressions"]]

total["'thickening' presence"] = [bool(re.search(r"thickening", x)) for x in total["impressions"]]
total["'thicken*' presence"] = [bool(re.search(r"thicken.*", x)) for x in total["impressions"]]
total["'abscess' presence"] = [bool(re.search(r"abscess", x)) for x in total["impressions"]]
total["'segmental mural hyperenhancement' presence"] = [bool(re.search(r"segmental mural hyperenhancement", x)) for x in total["impressions"]]
total["'asymmetric wall thickening' presence"] = [bool(re.search(r"asymmetric wall thickening", x)) for x in total["impressions"]]
total["'ulcerations' presence"] = [bool(re.search(r"ulcerations", x)) for x in total["impressions"]] 
total["'sacculations' presence"] = [bool(re.search(r"sacculations", x)) for x in total["impressions"]]
total["'intramural edema' presence"] = [bool(re.search(r"intramural edema", x)) for x in total["impressions"]]
total["'stricture' presence"] = [bool(re.search(r"stricture", x)) for x in total["impressions"]]
total["'simple fistula' presence"] = [bool(re.search(r"simple fistula", x)) for x in total["impressions"]]
total["'complex fistula' presence"] = [bool(re.search(r"complex fistula", x)) for x in total["impressions"]]
total["'sinus tract' presence"] = [bool(re.search(r"sinus tract", x)) for x in total["impressions"]]
total["'perianal fistula' presence"] = [bool(re.search(r"perianal fistula", x)) for x in total["impressions"]]
total["'inflammatory mass' presence"] = [bool(re.search(r"inflammatory mass", x)) for x in total["impressions"]]

total["'perienteric edema/inflammation' presence"] = [bool(re.search(r"perienteric edema|perienteric inflammation", x)) for x in total["impressions"]]
total["'engorged vasa recta' presence"] = [bool(re.search(r"engorged vasa recta", x)) for x in total["impressions"]]
total["'fibrofatty proliferation' presence"] = [bool(re.search(r"fibrofatty proliferation", x)) for x in total["impressions"]]
total["'mesenteric venous thrombosis/ occlusion' presence"] = [bool(re.search(r"mesenteric venous thrombosis|mesenteric venous occlusion", x)) for x in total["impressions"]]
total["'adenopathy' presence"] = [bool(re.search(r"adenopathy", x)) for x in total["impressions"]]
total["'obstruction' presence"] = [bool(re.search(r"obstruction", x)) for x in total["impressions"]]
total["'free' presence"] = [bool(re.search(r"free", x)) for x in total["impressions"]]
total["'colitis' presence"] = [bool(re.search(r"colitis", x)) for x in total["impressions"]]
total["'evidence' presence"] = [bool(re.search(r"evidence", x)) for x in total["impressions"]]
total["'increased' presence"] = [bool(re.search(r"increased", x)) for x in total["impressions"]]
total["'crohn' presence"] = [bool(re.search(r"crohn", x)) for x in total["impressions"]]
total["'mucosal' presence"] = [bool(re.search(r"mucosal", x)) for x in total["impressions"]]
total["'infiltration' presence"] = [bool(re.search(r"infiltration", x)) for x in total["impressions"]]
total["'pericolonic' presence"] = [bool(re.search(r"pericolonic", x)) for x in total["impressions"]]
total["'active' presence"] = [bool(re.search(r"active", x)) for x in total["impressions"]]
total["'mild' presence"] = [bool(re.search(r"mild", x)) for x in total["impressions"]]

#uncertainty
total["'no' presence (Negation)"] = [bool(re.search(r"\sno\s", x)) for x in total["impressions"]]
total["'not' presence (Negation)"] = [bool(re.search(r"\snot\s", x)) for x in total["impressions"]]
total["'no' in same sentence as 'acute' presence"] = [bool(re.search(r"(?=.*\bno\b)(?=.*\bacute\b)", x)) for x in total["impressions"]]
total["'no' in same sentence as 'chronic' presence"] = [bool(re.search(r"(?=.*\bno\b)(?=.*\bchronic\b)", x) ) for x in total["impressions"]]
total["'suspected' presence"] = [bool(re.search(r"suspected", x)) for x in total["impressions"]]
total["'query' presence"] = [bool(re.search(r"query", x)) for x in total["impressions"]]
total["'possible' presence"] = [bool(re.search(r"possible", x)) for x in total["impressions"]]
total["'rule out' presence"] = [bool(re.search(r"rule out", x)) for x in total["impressions"]]
total["'borderline' presence"] = [bool(re.search(r"borderline", x)) for x in total["impressions"]]
total["'indeterminate' presence"] = [bool(re.search(r"indeterminate", x)) for x in total["impressions"]]
total["'no definite' presence"] = [bool(re.search(r"no definite", x)) for x in total["impressions"]]
total["'probable' presence"] = [bool(re.search(r"probable", x)) for x in total["impressions"]]
total["'no obvious' presence"] = [bool(re.search(r"no obvious", x)) for x in total["impressions"]]
total["'equivocal' presence"] = [bool(re.search(r"equivocal", x)) for x in total["impressions"]]
total["'suggested|suggests' presence"] = [bool(re.search(r"suggested|suggests", x)) for x in total["impressions"]]

In [28]:
total.to_csv("recent_explodedimps.csv")

In [29]:
total

,mrn,service_date,impressions,'inflamm*' presence,'fat stranding' presence,'fat stra*' presence,'fat*' presence,'acute' presence,'fistula' presence,'enhancement' presence,...,'query' presence,'possible' presence,'rule out' presence,'borderline' presence,'indeterminate' presence,'no definite' presence,'probable' presence,'no obvious' presence,'equivocal' presence,'suggested|suggests' presence
0,12181,2013-08-18 00:00:00.000,Dilated proximal and decompressed distal small...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
0,12181,2013-08-18 00:00:00.000,Small ascites,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
0,12181,2013-08-18 00:00:00.000,Incompletely visualized lingular consolidation...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
0,12181,2013-08-18 00:00:00.000,Note: These findings were discussed with Dr. ...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,68012,2007-12-21 00:00:00.000,No CT findings to explain the patient's histo...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13471,76384401,2013-01-24 00:00:00.000,No evidence of recurrence or metastatic disea...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
13472,76384401,2013-06-03 00:00:00.000,No evidence for recurrence or metastatic disea...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
13473,76384401,2013-11-21 00:00:00.000,No evidence for recurrence or metastatic dise...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
13474,76384401,2014-05-12 00:00:00.000,No evidence of metastatic disease in the abdo...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [30]:
from collections import Counter

In [31]:

def tophundred_withcond(note):
    a = ""
    for index, row in note.iterrows():
        count = row["'inflamm*' presence"] + row["'fat stra*' presence"] + row["'fat*' presence"] + row["'acute' presence"] + row["'chronic' presence"] + row["'no' in same sentence as 'acute' presence"] + row["'no' in same sentence as 'chronic' presence"] 
        if count > 1:
            a += " " + row["impressions"]
    # split() returns list of all the words in the string 
    split_it = a.split()
    # Pass the split_it list to instance of Counter class. 
    counter = Counter(split_it) 
  
    # most_common() produces k frequently encountered 
    # input values and their respective counts. 
    most_occur = counter.most_common(100) 
    return most_occur
#tophundred_withcond(total)


In [32]:
def tophundred_gen(note):
   return Counter(" ".join(note["impressions"]).split()).most_common(150)


In [33]:
def relevancy(df):
    col_list = list(df)
    col_list.remove('mrn')
    col_list.remove('impressions')
    col_list.remove('service_date')
    df['relevancy (total features = true)'] = df[col_list].sum(axis=1)
    return df
relevancy(total)

,mrn,service_date,impressions,'inflamm*' presence,'fat stranding' presence,'fat stra*' presence,'fat*' presence,'acute' presence,'fistula' presence,'enhancement' presence,...,'possible' presence,'rule out' presence,'borderline' presence,'indeterminate' presence,'no definite' presence,'probable' presence,'no obvious' presence,'equivocal' presence,'suggested|suggests' presence,relevancy (total features = true)
0,12181,2013-08-18 00:00:00.000,Dilated proximal and decompressed distal small...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1
0,12181,2013-08-18 00:00:00.000,Small ascites,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0
0,12181,2013-08-18 00:00:00.000,Incompletely visualized lingular consolidation...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0
0,12181,2013-08-18 00:00:00.000,Note: These findings were discussed with Dr. ...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0
3,68012,2007-12-21 00:00:00.000,No CT findings to explain the patient's histo...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13471,76384401,2013-01-24 00:00:00.000,No evidence of recurrence or metastatic disea...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1
13472,76384401,2013-06-03 00:00:00.000,No evidence for recurrence or metastatic disea...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1
13473,76384401,2013-11-21 00:00:00.000,No evidence for recurrence or metastatic dise...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1
13474,76384401,2014-05-12 00:00:00.000,No evidence of metastatic disease in the abdo...,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1


In [34]:
def relevancy_filter(note):
    df = note[note["relevancy (total features = true)"] != 0]
    return df
relevancyfilter = relevancy_filter(total)
relevancyfilter.to_csv('relevancy_filter.csv')


In [35]:
subset_pos = [relevancyfilter.iloc[5], relevancyfilter.iloc[95], relevancyfilter.iloc[29], 
             relevancyfilter.iloc[54], relevancyfilter.iloc[75], relevancyfilter.iloc[291], relevancyfilter.iloc[1246],
             relevancyfilter.iloc[56], relevancyfilter.iloc[235], relevancyfilter.iloc[396], 
             relevancyfilter.iloc[576], relevancyfilter.iloc[738], relevancyfilter.iloc[45],
             relevancyfilter.iloc[3482], relevancyfilter.iloc[334], relevancyfilter.iloc[729], relevancyfilter.iloc[1018],
             relevancyfilter.iloc[2404], relevancyfilter.iloc[767], relevancyfilter.iloc[6222]]

subset_neg = [relevancyfilter.iloc[706], relevancyfilter.iloc[1096], relevancyfilter.iloc[1294], relevancyfilter.iloc[2073],
             relevancyfilter.iloc[3596], relevancyfilter.iloc[2849], relevancyfilter.iloc[1418], relevancyfilter.iloc[5026],
             relevancyfilter.iloc[3596], relevancyfilter.iloc[5973], relevancyfilter.iloc[1359], 
             relevancyfilter.iloc[6080], relevancyfilter.iloc[7343], relevancyfilter.iloc[2878], relevancyfilter.iloc[108],
             relevancyfilter.iloc[343], relevancyfilter.iloc[2437], relevancyfilter.iloc[3145], 
             relevancyfilter.iloc[5946], relevancyfilter.iloc[5211], relevancyfilter.iloc[5355]]

neg_df = pd.DataFrame(subset_neg)
pos_df = pd.DataFrame(subset_pos)


In [36]:
relevancyfilter.iloc[706]

mrn                                                                            9340815
service_date                                                   2012-10-11 00:00:00.000
impressions                          Cholelithiasis with no evidence for acute infl...
'inflamm*' presence                                                               True
'fat stranding' presence                                                         False
                                                           ...                        
'probable' presence                                                              False
'no obvious' presence                                                            False
'equivocal' presence                                                             False
'suggested|suggests' presence                                                    False
relevancy (total features = true)                                                    5
Name: 1288, Length: 66, dtype: object

In [37]:
def inferred_pos(note):
    df = note[["mrn", "service_date", "impressions"]].copy()
    notecopy = note[["mrn", "service_date", "impressions", "'inflamm*' presence",
                     "'fat stra*' presence", "'fistula' presence", "'enhancement' presence",
                     "'hyperemia' presence", "'wall thickening' presence", "'abscess' presence", "'thicken*' presence",
                     "'segmental mural hyperenhancement' presence", "'asymmetric wall thickening' presence",
                     "'ulcerations' presence", "'stricture' presence", "'sinus tract' presence", "'hyperenhancement' presence", "'thickening' presence",
                     "'active' presence", "'suspected' presence", "'colitis' presence"]]
    col_list = list(notecopy)
    col_list.remove('mrn')
    col_list.remove('impressions')
    col_list.remove('service_date')
    notecopy["Prediction"] =  notecopy[col_list].sum(axis=1)
    #predictions = []
    #for x in notecopy["Prediction"]:
        #if x > 0:
         #   predictions.append("Active")
        #if x == 0:
         #   predictions.append("Inactive")
    df["predictions"] = ["Active" if x > 0 else "Inactive" for x in notecopy["Prediction"]]
    return df
b = inferred_pos(pos_df)
b.to_csv("inferred_pos.csv")

/opt/ars-virtenv/lib/python3.6/site-packages/ipykernel_launcher.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  del sys.path[0]


In [38]:
def inferred_neg(note):
    df = note[["mrn", "service_date", "impressions"]].copy()
    notecopy = note[["mrn", "service_date", "impressions", "'no' in same sentence as 'acute' presence",
                    "'no' in same sentence as 'chronic' presence", "'no definite' presence",
                     "'no obvious' presence"]]
    col_list = list(notecopy)
    col_list.remove('mrn')
    col_list.remove('impressions')
    col_list.remove('service_date')
    notecopy["Prediction"] =  notecopy[col_list].sum(axis=1)
    #predictions = []
    #for x in notecopy["Prediction"]:
        #if x > 0:
         #   predictions.append("Active")
        #if x == 0:
         #   predictions.append("Inactive")
    df["predictions"] = ["Inactive" if x > 0 else "Active" for x in notecopy["Prediction"]]
    return df
c = inferred_neg(neg_df)
c.to_csv("inferred_neg.csv")

/opt/ars-virtenv/lib/python3.6/site-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.


In [39]:
neg_with_pred = neg_df.copy()
neg_with_pred['predictions'] = c['predictions']
pos_with_pred = pos_df.copy()
pos_with_pred['predictions'] = b['predictions']
neg_pos = [neg_with_pred, pos_with_pred]
predictions_df = pd.concat(neg_pos)
predictions_df.to_csv('predictions_df.csv')

In [40]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
#Import scikit-learn metrics module for accuracy calculation
from sklearn import metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

In [41]:
X = predictions_df[["'inflamm*' presence",
                     "'fat stra*' presence", "'fistula' presence", "'enhancement' presence",
                     "'hyperemia' presence", "'wall thickening' presence", "'abscess' presence", "'thicken*' presence",
                     "'segmental mural hyperenhancement' presence", "'asymmetric wall thickening' presence",
                     "'ulcerations' presence", "'stricture' presence", "'sinus tract' presence", "'hyperenhancement' presence", "'thickening' presence",
                     "'active' presence", "'suspected' presence", "'colitis' presence",  "'no' in same sentence as 'acute' presence",
                    "'no' in same sentence as 'chronic' presence", "'no definite' presence",
                     "'no obvious' presence"]]
y = predictions_df[['predictions']]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [ ]:
clf=RandomForestClassifier(n_estimators=100)

#Train the model using the training sets y_pred=clf.predict(X_test)
clf.fit(X_train,y_train)

y_pred=clf.predict(X_test)

# Model Accuracy, how often is the classifier correct?
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))

In [ ]:
#feature_imp = pd.Series(clf.feature_importances_,index=predictions_df.feature_names).sort_values(ascending=False)
#feature_imp

In [ ]:
clf.predict()